In [2]:
# ============================================================
# OVERNIGHT RETRAIN: All λ values with v2 strategy
# (paraphrase-warmstart + length + ROUGE-floor)
#
# Run this as a SINGLE NOTEBOOK — paste all cells below.
# Total time: ~10-12 hrs on T4 (5 λ values × ~2 hrs each)
#
# Checkpoints saved to Drive after each λ — safe to resume.
# ============================================================
 
 
# ── CELL 1: Setup ─────────────────────────────────────────────
# from google.colab import drive
# drive.mount('/content/drive')
 
SAVE_DIR = "NLP_λ_Sweep_Checkpoints"
import os; os.makedirs(SAVE_DIR, exist_ok=True)

In [3]:
!pip install transformers peft datasets huggingface_hub bert-score rouge-score accelerate sentencepiece protobuf -q

In [4]:
# ── CELL 2: Imports ───────────────────────────────────────────
import json, random, os, gc
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from bert_score import score as bert_score_fn
from rouge_score import rouge_scorer as rouge_lib
from huggingface_hub import hf_hub_download
from peft import LoraConfig, TaskType, get_peft_model
from tqdm import tqdm
from transformers import (
    AutoModelForSequenceClassification, AutoTokenizer,
    T5ForConditionalGeneration, AutoModel,
    AutoModelForCausalLM,
)
 
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
torch.manual_seed(42)
random.seed(42)

/home/student/miniforge3/envs/llm_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


In [5]:
# ── CELL 3: Load HC3 ─────────────────────────────────────────
filepath = hf_hub_download(
    repo_id="Hello-SimpleAI/HC3",
    filename="all.jsonl",
    repo_type="dataset",
)
rows = [json.loads(line) for line in open(filepath)]
ai_texts = []
for r in rows:
    for a in r.get("chatgpt_answers", []):
        if len(a.strip()) > 50:
            ai_texts.append(a.strip())
random.shuffle(ai_texts)
 
train_ai = ai_texts[:5000]
test_ai  = ai_texts[5000:5200]
print(f"Train: {len(train_ai)}  Test: {len(test_ai)}")

Train: 5000  Test: 200


In [6]:
# ── CELL 4: Load detector ────────────────────────────────────
DET_NAME = "Hello-SimpleAI/chatgpt-detector-roberta"
det_tok  = AutoTokenizer.from_pretrained(DET_NAME)
detector = AutoModelForSequenceClassification.from_pretrained(DET_NAME).to(device)
detector.eval()
for p in detector.parameters():
    p.requires_grad_(False)
print("Detector loaded.")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 21968.86it/s]
RobertaForSequenceClassification LOAD REPORT from: Hello-SimpleAI/chatgpt-detector-roberta
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Detector loaded.


In [7]:
# ── CELL 5: Build model (v2 — paraphrase-warmstarted) ────────
def build_model_v2(base="Vamsi/T5_Paraphrase_Paws"):
    tokenizer  = AutoTokenizer.from_pretrained(base, use_fast=False)
    base_model = T5ForConditionalGeneration.from_pretrained(base)
    lora_cfg   = LoraConfig(
        task_type=TaskType.SEQ_2_SEQ_LM,
        r=16, lora_alpha=32, lora_dropout=0.05,
        target_modules=["q", "v"],
    )
    model = get_peft_model(base_model, lora_cfg).to(device)
    model.print_trainable_parameters()
    return model, tokenizer

In [8]:
# ── CELL 6: Loss functions ───────────────────────────────────
print("Loading MiniLM for semantic loss...")
sem_tok   = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
sem_model = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2").to(device)
sem_model.eval()
for p in sem_model.parameters():
    p.requires_grad_(False)
print("MiniLM loaded.")
 
def mean_pool(out, mask):
    t = out.last_hidden_state
    m = mask.unsqueeze(-1).expand(t.size()).float()
    return (t * m).sum(1) / m.sum(1).clamp(min=1e-9)
 
def compute_l_grad(token_logits):
    det_embed = detector.roberta.embeddings.word_embeddings.weight
    probs     = F.softmax(token_logits, dim=-1)
    min_v     = min(probs.shape[-1], det_embed.shape[0])
    pseudo    = torch.matmul(probs[:, :, :min_v], det_embed[:min_v])
    logits    = detector(inputs_embeds=pseudo).logits
    targets   = torch.zeros(logits.size(0), dtype=torch.long, device=device)
    return F.cross_entropy(logits, targets)
 
def compute_l_rl(model, tokenizer, texts, G=2, max_new=48):
    loss_accum = torch.tensor(0.0, device=device)
    for text in texts[:1]:
        inp = tokenizer(
            "paraphrase: " + text[:300],
            return_tensors="pt", truncation=True, max_length=96
        ).to(device)
        lps, rewards = [], []
        for _ in range(G):
            with torch.no_grad():
                out = model.generate(
                    input_ids=inp["input_ids"],
                    attention_mask=inp["attention_mask"],
                    max_new_tokens=max_new,
                    do_sample=True, temperature=0.9,
                    output_scores=True,
                    return_dict_in_generate=True,
                    decoder_start_token_id=0
                )
            ids  = out.sequences[0][inp.input_ids.shape[1]:]
            if len(ids) == 0:
                continue
            lp = 0.0
            for i, s in enumerate(out.scores):
                if i >= len(ids): break
                token_lp = F.log_softmax(s, dim=-1)[0, ids[i]].item()
                if not (torch.isnan(torch.tensor(token_lp)) or
                        torch.isinf(torch.tensor(token_lp))):
                    lp += token_lp
            cand = tokenizer.decode(ids, skip_special_tokens=True)
            if not cand.strip():
                continue
            enc = det_tok(cand, return_tensors="pt",
                          truncation=True, max_length=96).to(device)
            with torch.no_grad():
                ai_p = torch.softmax(detector(**enc).logits, dim=-1)[0, 1].item()
            lps.append(lp)
            rewards.append(1.0 - ai_p)
 
        if len(lps) < 2:
            continue
        r_t  = torch.tensor(rewards, dtype=torch.float32, device=device)
        lp_t = torch.tensor(lps,     dtype=torch.float32, device=device)
        lp_t = torch.clamp(lp_t, min=-50.0, max=0.0)
        adv  = (r_t - r_t.mean()) / (r_t.std() + 1e-8)
        loss_accum = loss_accum - (adv * lp_t).mean()
 
    if torch.isnan(loss_accum) or torch.isinf(loss_accum):
        return torch.tensor(0.0, device=device)
    return loss_accum
 
def compute_l_sem(originals, paraphrases):
    enc_o = sem_tok(originals,   padding=True, truncation=True,
                    max_length=128, return_tensors="pt").to(device)
    enc_p = sem_tok(paraphrases, padding=True, truncation=True,
                    max_length=128, return_tensors="pt").to(device)
    with torch.no_grad():
        emb_o = mean_pool(sem_model(**enc_o), enc_o["attention_mask"])
        emb_p = mean_pool(sem_model(**enc_p), enc_p["attention_mask"])
    return (1.0 - F.cosine_similarity(emb_o, emb_p).mean()).to(device)
 
def compute_l_len(orig_ids_len, gen_ids_len):
    ratio = abs(gen_ids_len - orig_ids_len) / max(orig_ids_len, 1)
    return torch.tensor(min(ratio ** 2, 4.0), device=device)
 
def compute_l_rouge_floor(originals, paraphrases, floor=0.25):
    sc = rouge_lib.RougeScorer(["rougeL"], use_stemmer=True)
    scores = [sc.score(o, p)["rougeL"].fmeasure
              for o, p in zip(originals, paraphrases)]
    avg = float(np.mean(scores))
    return torch.tensor(max(0.0, floor - avg), device=device)
 
print("Loss functions ready.")

Loading MiniLM for semantic loss...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3649.41it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


MiniLM loaded.
Loss functions ready.


In [9]:
# ── CELL 7: Eval helpers ─────────────────────────────────────
def generate_paraphrases(model, tokenizer, texts, max_new=128, batch_size=8):
    model.eval()
    results = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc   = tokenizer(
            ["paraphrase: " + t for t in batch],
            return_tensors="pt", padding=True,
            truncation=True, max_length=256
        ).to(device)
        in_len = enc["input_ids"].shape[1]
        with torch.no_grad():
            ids = model.generate(
                input_ids=enc["input_ids"],
                attention_mask=enc["attention_mask"],
                max_new_tokens=in_len,
                min_new_tokens=int(0.7 * in_len),
                num_beams=4, no_repeat_ngram_size=3,
                do_sample=False,
                decoder_start_token_id=0,
            )
        results += tokenizer.batch_decode(ids, skip_special_tokens=True)
    return results
 
def eval_asr(texts, batch_size=32):
    preds = []
    for i in range(0, len(texts), batch_size):
        enc = det_tok(texts[i:i+batch_size], return_tensors="pt",
                      padding=True, truncation=True, max_length=512).to(device)
        with torch.no_grad():
            preds += torch.argmax(detector(**enc).logits, dim=-1).cpu().tolist()
    return sum(p == 0 for p in preds) / len(preds) * 100
 
def eval_bertscore(originals, paraphrases):
    try:
        from bert_score import score as bs_fn
        _, _, F1 = bs_fn(
            paraphrases, originals,
            model_type="distilbert-base-uncased",
            device=device, verbose=False
        )
        return F1.mean().item()
    except Exception as e:
        print(f"  BERTScore skipped: {e}")
        return -1.0
 
def eval_rouge(originals, paraphrases):
    sc = rouge_lib.RougeScorer(["rougeL"], use_stemmer=True)
    return np.mean([sc.score(o, p)["rougeL"].fmeasure
                    for o, p in zip(originals, paraphrases)])
 
print("Eval helpers ready.")

Eval helpers ready.


In [10]:
# ── CELL 8: v2 training function ─────────────────────────────
def train_one_lambda_v2(
    lam,
    alpha        = 0.5,
    beta_len     = 0.5,
    beta_rouge   = 1.0,
    epochs       = 3,
    batch_size   = 4,
    rl_per_batch = 1,
    max_train    = 5000,
):
    lam_tag   = str(lam).replace(".", "_") + "_v2"
    save_path = f"{SAVE_DIR}/lambda_{lam_tag}"
    os.makedirs(save_path, exist_ok=True)
 
    done_file = f"{save_path}/DONE.json"
    if os.path.exists(done_file):
        print(f"λ={lam} v2 already done — skipping.")
        with open(done_file) as f:
            return json.load(f)
 
    print(f"\n{'='*55}")
    print(f"  λ = {lam}  (v2: paraphrase-warmstart)")
    print(f"{'='*55}")
 
    model, tokenizer = build_model_v2()
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad], lr=2e-4
    )
 
    train_data = train_ai[:max_train]
 
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        random.shuffle(train_data)
 
        for i in tqdm(range(0, len(train_data), batch_size),
                      desc=f"  Epoch {epoch+1}/{epochs}  λ={lam}"):
            batch = train_data[i:i+batch_size]
            enc   = tokenizer(
                ["paraphrase: " + t for t in batch],
                return_tensors="pt", padding=True,
                truncation=True, max_length=128
            ).to(device)
 
            out = model(
                input_ids=enc["input_ids"],
                attention_mask=enc["attention_mask"],
                decoder_input_ids=enc["input_ids"],
            )
 
            loss = torch.tensor(0.0, device=device)
            if lam > 0:
                loss = loss + lam * compute_l_grad(out.logits)
            if lam < 1.0:
                loss = loss + (1 - lam) * compute_l_rl(
                    model, tokenizer, batch[:rl_per_batch]
                )
 
            input_len = int(enc["input_ids"].shape[1])
            with torch.no_grad():
                gen_ids = model.generate(
                    input_ids=enc["input_ids"],
                    attention_mask=enc["attention_mask"],
                    max_new_tokens=input_len,
                    min_new_tokens=int(0.7 * input_len),
                    do_sample=False,
                    decoder_start_token_id=0,
                    no_repeat_ngram_size=3,
                )
            paras = tokenizer.batch_decode(gen_ids, skip_special_tokens=True)
 
            loss = loss + alpha * compute_l_sem(batch, paras)
            loss = loss + beta_len * compute_l_len(input_len, gen_ids.shape[1])
            loss = loss + beta_rouge * compute_l_rouge_floor(batch, paras)
 
            optimizer.zero_grad()
            torch.cuda.empty_cache()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            epoch_loss += loss.item()
 
        n = max(1, len(train_data) // batch_size)
        print(f"  Epoch {epoch+1} | Loss: {epoch_loss/n:.4f}")
        ckpt = f"{save_path}/epoch_{epoch+1}"
        model.save_pretrained(ckpt); tokenizer.save_pretrained(ckpt)
 
    # ── Evaluate ──────────────────────────────────────────────
    print(f"\nEvaluating λ={lam} v2 on 200 test samples...")
    paras = generate_paraphrases(model, tokenizer, test_ai[:200])
 
    asr = eval_asr(paras)
    bs  = eval_bertscore(test_ai[:200], paras)
    rl  = eval_rouge(test_ai[:200], paras)
 
    pd.DataFrame({
        "id":              [f"hc3_ai_{i:05d}_evaded" for i in range(len(paras))],
        "text":            paras,
        "source":          "ai",
        "attack_type":     "gradient",
        "attack_owner":    "udaiveer",
        "generator_model": "gpt3.5-turbo",
        "original_text":   test_ai[:200],
    }).to_csv(f"{save_path}/evaded.csv", index=False)
 
    results = {
        "lambda": lam, "variant": "v2",
        "asr": round(asr, 2),
        "bertscore_f1": round(bs, 4),
        "rouge_l": round(rl, 4),
        "loss_final": round(epoch_loss / n, 4),
    }
    with open(done_file, "w") as f:
        json.dump(results, f, indent=2)
    print(f"  λ={lam} v2 DONE → ASR:{asr:.1f}%  BS:{bs:.4f}  RL:{rl:.4f}")
 
    # Cleanup VRAM
    del model, tokenizer, optimizer
    gc.collect()
    torch.cuda.empty_cache()
 
    return results

In [12]:
# ── FIX: Patch train_one_lambda_v2 for λ=0.0 ─────────────────
# The issue is that compute_l_rl returns a detached tensor when
# all candidates fail or rewards < 2. We need to ensure the loss
# always has grad_fn by routing through a differentiable path.

_original_train = train_one_lambda_v2  # keep reference

def train_one_lambda_v2_fixed(
    lam,
    alpha        = 0.5,
    beta_len     = 0.5,
    beta_rouge   = 1.0,
    epochs       = 3,
    batch_size   = 4,
    rl_per_batch = 1,
    max_train    = 5000,
):
    # For lam > 0, the original function works fine
    if lam > 0:
        return _original_train(lam, alpha, beta_len, beta_rouge,
                               epochs, batch_size, rl_per_batch, max_train)

    # λ=0.0: pure RL needs special handling
    lam_tag   = "0_0_v2"
    save_path = f"{SAVE_DIR}/lambda_{lam_tag}"
    os.makedirs(save_path, exist_ok=True)

    done_file = f"{save_path}/DONE.json"
    if os.path.exists(done_file):
        print(f"λ=0.0 v2 already done — skipping.")
        with open(done_file) as f:
            return json.load(f)

    print(f"\n{'='*55}")
    print(f"  λ = 0.0  (v2: pure RL, paraphrase-warmstart)")
    print(f"{'='*55}")

    model, tokenizer = build_model_v2()
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad], lr=2e-4
    )

    train_data = train_ai[:max_train]

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        random.shuffle(train_data)

        for i in tqdm(range(0, len(train_data), batch_size),
                      desc=f"  Epoch {epoch+1}/{epochs}  λ=0.0"):
            batch = train_data[i:i+batch_size]
            enc   = tokenizer(
                ["paraphrase: " + t for t in batch],
                return_tensors="pt", padding=True,
                truncation=True, max_length=128
            ).to(device)

            # ── Pure RL: need differentiable log probs ────────
            rl_loss = torch.tensor(0.0, device=device)
            has_grad = False

            for text in batch[:rl_per_batch]:
                inp = tokenizer(
                    "paraphrase: " + text[:300],
                    return_tensors="pt", truncation=True, max_length=96
                ).to(device)

                G = 2
                rewards, log_prob_tensors = [], []

                enc_out = model.model.encoder(
                    input_ids=inp["input_ids"],
                    attention_mask=inp["attention_mask"]
                )

                for _ in range(G):
                    cur_ids = torch.zeros(1, 1, dtype=torch.long, device=device)
                    token_log_probs = []
                    generated_ids = []

                    for step in range(48):
                        out = model.model.decoder(
                            input_ids=cur_ids,
                            encoder_hidden_states=enc_out.last_hidden_state,
                            encoder_attention_mask=inp["attention_mask"]
                        )
                        logits = model.lm_head(out.last_hidden_state[:, -1, :])
                        probs  = torch.softmax(logits / 0.9, dim=-1)
                        token  = torch.multinomial(probs, 1)
                        lp     = torch.log(probs[0, token.item()] + 1e-8)
                        token_log_probs.append(lp)
                        generated_ids.append(token.item())
                        cur_ids = token
                        if token.item() == tokenizer.eos_token_id:
                            break

                    cand = tokenizer.decode(generated_ids, skip_special_tokens=True)
                    if cand.strip() and token_log_probs:
                        enc_d = det_tok(cand, return_tensors="pt",
                                        truncation=True, max_length=96).to(device)
                        with torch.no_grad():
                            ai_p = torch.softmax(detector(**enc_d).logits, dim=-1)[0, 1].item()
                        rewards.append(1.0 - ai_p)
                        log_prob_tensors.append(torch.stack(token_log_probs).mean())

                if len(rewards) >= 2 and len(log_prob_tensors) >= 2:
                    r_t = torch.tensor(rewards, device=device)
                    adv = (r_t - r_t.mean()) / (r_t.std() + 1e-8)
                    for a, lp in zip(adv, log_prob_tensors):
                        rl_loss = rl_loss - a * lp
                    rl_loss = rl_loss / len(rewards)
                    has_grad = True

            # Semantic + length + rouge (these are detached, just scalars)
            input_len = int(enc["input_ids"].shape[1])
            with torch.no_grad():
                gen_ids = model.generate(
                    input_ids=enc["input_ids"],
                    attention_mask=enc["attention_mask"],
                    max_new_tokens=input_len,
                    min_new_tokens=int(0.7 * input_len),
                    do_sample=False,
                    decoder_start_token_id=0,
                    no_repeat_ngram_size=3,
                )
            paras = tokenizer.batch_decode(gen_ids, skip_special_tokens=True)

            sem_loss   = alpha * compute_l_sem(batch, paras)
            len_loss   = beta_len * compute_l_len(input_len, gen_ids.shape[1])
            rouge_loss = beta_rouge * compute_l_rouge_floor(batch, paras)

            total = rl_loss + sem_loss + len_loss + rouge_loss

            if has_grad and total.requires_grad:
                optimizer.zero_grad()
                torch.cuda.empty_cache()
                total.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            epoch_loss += total.item()

        n = max(1, len(train_data) // batch_size)
        print(f"  Epoch {epoch+1} | Loss: {epoch_loss/n:.4f}")
        ckpt = f"{save_path}/epoch_{epoch+1}"
        model.save_pretrained(ckpt); tokenizer.save_pretrained(ckpt)

    # Evaluate
    print(f"\nEvaluating λ=0.0 v2 on 200 test samples...")
    paras = generate_paraphrases(model, tokenizer, test_ai[:200])
    asr = eval_asr(paras)
    bs  = eval_bertscore(test_ai[:200], paras)
    rl  = eval_rouge(test_ai[:200], paras)

    pd.DataFrame({
        "id":              [f"hc3_ai_{i:05d}_evaded" for i in range(len(paras))],
        "text":            paras,
        "source":          "ai",
        "attack_type":     "gradient",
        "attack_owner":    "udaiveer",
        "generator_model": "gpt3.5-turbo",
        "original_text":   test_ai[:200],
    }).to_csv(f"{save_path}/evaded.csv", index=False)

    results = {
        "lambda": 0.0, "variant": "v2",
        "asr": round(asr, 2),
        "bertscore_f1": round(bs, 4),
        "rouge_l": round(rl, 4),
        "loss_final": round(epoch_loss / n, 4),
    }
    with open(done_file, "w") as f:
        json.dump(results, f, indent=2)
    print(f"  λ=0.0 v2 DONE → ASR:{asr:.1f}%  BS:{bs:.4f}  RL:{rl:.4f}")

    del model, tokenizer, optimizer
    gc.collect(); torch.cuda.empty_cache()
    return results

# Override the function
train_one_lambda_v2 = train_one_lambda_v2_fixed
print("Patched. Now re-run the sweep cell — it will skip 1.0/0.75/0.5/0.25 and only train 0.0.")

Patched. Now re-run the sweep cell — it will skip 1.0/0.75/0.5/0.25 and only train 0.0.


In [13]:
# ── CELL 9: Run full v2 sweep ─────────────────────────────────
LAMBDA_VALUES = [1.0, 0.75, 0.5, 0.25, 0.0]
all_results_v2 = []
 
for lam in LAMBDA_VALUES:
    res = train_one_lambda_v2(
        lam        = lam,
        alpha      = 0.5,
        beta_len   = 0.5,
        beta_rouge = 1.0,
        epochs     = 3,
        batch_size = 4,
        rl_per_batch = 1,
        max_train  = 5000,
    )
    all_results_v2.append(res)
    gc.collect()
    torch.cuda.empty_cache()
 
summary_v2 = pd.DataFrame(all_results_v2).sort_values("lambda")
summary_v2.to_csv(f"{SAVE_DIR}/lambda_sweep_v2_results.csv", index=False)
print("\n" + "="*55)
print("V2 SWEEP COMPLETE")
print("="*55)
print(summary_v2.to_string(index=False))

λ=1.0 v2 already done — skipping.
λ=0.75 v2 already done — skipping.
λ=0.5 v2 already done — skipping.
λ=0.25 v2 already done — skipping.

  λ = 0.0  (v2: pure RL, paraphrase-warmstart)


Loading weights: 100%|██████████| 257/257 [00:00<00:00, 3531.74it/s]


trainable params: 1,769,472 || all params: 224,673,024 || trainable%: 0.7876


  Epoch 1/3  λ=0.0: 100%|██████████| 1250/1250 [1:00:20<00:00,  2.90s/it]


  Epoch 1 | Loss: 0.0310


...

  Epoch 2 | Loss: 0.0283


  Epoch 3/3  λ=0.0: 100%|██████████| 1250/1250 [2:54:16<00:00,  8.37s/it] 


  Epoch 3 | Loss: 0.0292

Evaluating λ=0.0 v2 on 200 test samples...


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 3979.79it/s]
DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  λ=0.0 v2 DONE → ASR:100.0%  BS:0.8988  RL:0.6413

V2 SWEEP COMPLETE
 lambda variant   asr  bertscore_f1  rouge_l  loss_final
   0.00      v2 100.0        0.8988   0.6413      0.0292
   0.25      v2  98.5        0.8519   0.4930      0.0789
   0.50      v2  99.5        0.8317   0.4592         NaN
   0.75      v2  97.0        0.8548   0.5035      0.0721
   1.00      v2  99.0        0.8809   0.6362      0.0421


In [14]:
# ── CELL 10: Score ALL v2 models against DetectGPT + Fast-DGPT ─
 
print("\n\nLoading scoring models for CPTR...")
 
# DetectGPT models
score_tokenizer = AutoTokenizer.from_pretrained("gpt2")
score_model_gpt = AutoModelForCausalLM.from_pretrained("gpt2").to(device)
score_model_gpt.eval()
 
t5_tok_small = AutoTokenizer.from_pretrained("t5-small", use_fast=False)
t5_model_small = T5ForConditionalGeneration.from_pretrained("t5-small").to(device)
t5_model_small.eval()
 
# Fast-DetectGPT models
score_tok_m   = AutoTokenizer.from_pretrained("gpt2-medium")
score_model_m = AutoModelForCausalLM.from_pretrained("gpt2-medium").to(device)
score_model_m.eval()
 
ref_tok   = AutoTokenizer.from_pretrained("gpt2")
ref_model = AutoModelForCausalLM.from_pretrained("gpt2").to(device)
ref_model.eval()
 
 
def get_log_prob(text, max_len=128):
    inputs = score_tokenizer(text, return_tensors="pt",
                             truncation=True, max_length=max_len).to(device)
    with torch.no_grad():
        loss = score_model_gpt(**inputs, labels=inputs["input_ids"]).loss
    return -loss.item()
 
def perturb_text(text, n=10):
    words = text.split()
    if len(words) < 5:
        return [text] * n
    results = []
    for _ in range(n):
        masked = words.copy()
        idx = random.sample(range(len(masked)), max(1, int(len(masked)*0.15)))
        for i in idx:
            masked[i] = "<extra_id_0>"
        masked_text = " ".join(masked)
        try:
            inp = t5_tok_small(masked_text, return_tensors="pt",
                               truncation=True, max_length=128).to(device)
            with torch.no_grad():
                out = t5_model_small.generate(**inp, max_new_tokens=15)
            filled = t5_tok_small.decode(out[0], skip_special_tokens=True)
            fill_word = filled.strip().split()[0] if filled.strip() else words[idx[0]]
            result = [w if w != "<extra_id_0>" else fill_word for w in masked]
            results.append(" ".join(result))
        except:
            results.append(text)
    return results
 
def detectgpt_score(text, n=10):
    orig_lp = get_log_prob(text)
    perturbs = perturb_text(text, n)
    perturb_lps = [get_log_prob(p) for p in perturbs]
    return orig_lp - float(np.mean(perturb_lps))
 
def get_token_log_probs(text, tokenizer, model, max_len=256):
    inputs = tokenizer(text, return_tensors="pt",
                       truncation=True, max_length=max_len).to(device)
    with torch.no_grad():
        out    = model(**inputs, labels=inputs["input_ids"])
        logits = out.logits[:, :-1, :]
        labels = inputs["input_ids"][:, 1:]
        lp     = F.log_softmax(logits, dim=-1)
        token_lps = lp.gather(-1, labels.unsqueeze(-1)).squeeze(-1)
    return token_lps.mean().item()
 
def fast_detectgpt_score(text):
    lp_score = get_token_log_probs(text, score_tok_m, score_model_m)
    lp_ref   = get_token_log_probs(text, ref_tok, ref_model)
    return lp_score - lp_ref
 
 
DGPT_THRESHOLD  = 1.039
FDGPT_THRESHOLD = 0.0
SCORE_N         = 100
 
all_scores = []
 
for lam in LAMBDA_VALUES:
    lam_tag     = str(lam).replace(".", "_") + "_v2"
    evaded_path = f"{SAVE_DIR}/lambda_{lam_tag}/evaded.csv"
 
    if not os.path.exists(evaded_path):
        print(f"λ={lam} v2: no evaded.csv, skipping scoring")
        continue
 
    evaded_df = pd.read_csv(evaded_path)
    texts = [str(t) for t in evaded_df["text"].tolist()
             if str(t).strip() and len(str(t)) > 30][:SCORE_N]
 
    if len(texts) < 10:
        print(f"λ={lam} v2: too few valid texts ({len(texts)}), skipping")
        continue
 
    print(f"\n--- Scoring λ={lam} v2 ({len(texts)} texts) ---")
 
    # DetectGPT
    dgpt_scores = [detectgpt_score(t) for t in tqdm(texts, desc="DetectGPT")]
    dgpt_fooled = sum(s < DGPT_THRESHOLD for s in dgpt_scores)
    dgpt_cptr   = dgpt_fooled / len(dgpt_scores) * 100
 
    # Fast-DetectGPT
    fdgpt_scores = [fast_detectgpt_score(t) for t in tqdm(texts, desc="Fast-DGPT")]
    fdgpt_fooled = sum(s < FDGPT_THRESHOLD for s in fdgpt_scores)
    fdgpt_cptr   = fdgpt_fooled / len(fdgpt_scores) * 100
 
    row = {
        "lambda": lam,
        "cptr_detectgpt": round(dgpt_cptr, 2),
        "cptr_fastdgpt":  round(fdgpt_cptr, 2),
        "avg_dgpt_score": round(float(np.mean(dgpt_scores)), 4),
        "avg_fdgpt_score": round(float(np.mean(fdgpt_scores)), 4),
    }
    all_scores.append(row)
    print(f"  λ={lam} → DGPT CPTR: {dgpt_cptr:.1f}%  FDGPT CPTR: {fdgpt_cptr:.1f}%")
 
scores_df = pd.DataFrame(all_scores)
scores_df.to_csv(f"{SAVE_DIR}/cptr_all_v2.csv", index=False)



Loading scoring models for CPTR...


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 4361.85it/s]



--- Scoring λ=1.0 v2 (100 texts) ---


Fast-DGPT: 100%|██████████| 100/100 [00:02<00:00, 39.26it/s]


  λ=1.0 → DGPT CPTR: 79.0%  FDGPT CPTR: 0.0%

--- Scoring λ=0.75 v2 (100 texts) ---


...

  λ=0.5 → DGPT CPTR: 92.0%  FDGPT CPTR: 3.0%

--- Scoring λ=0.25 v2 (100 texts) ---


Fast-DGPT: 100%|██████████| 100/100 [00:02<00:00, 39.46it/s]


  λ=0.25 → DGPT CPTR: 96.0%  FDGPT CPTR: 9.0%

--- Scoring λ=0.0 v2 (100 texts) ---


Fast-DGPT: 100%|██████████| 100/100 [00:02<00:00, 40.04it/s]

  λ=0.0 → DGPT CPTR: 55.0%  FDGPT CPTR: 0.0%


In [15]:
# ── CELL 11: Merge everything into final results table ────────
summary_v2 = pd.read_csv(f"{SAVE_DIR}/lambda_sweep_v2_results.csv")
scores_df  = pd.read_csv(f"{SAVE_DIR}/cptr_all_v2.csv")
final = summary_v2.merge(scores_df, on="lambda", how="left")
final.to_csv(f"{SAVE_DIR}/final_results_v2.csv", index=False)
 
print("\n" + "="*55)
print("FINAL RESULTS TABLE (v2)")
print("="*55)
print(final.to_string(index=False))
print(f"\nSaved → {SAVE_DIR}/final_results_v2.csv")


FINAL RESULTS TABLE (v2)
 lambda variant   asr  bertscore_f1  rouge_l  loss_final  cptr_detectgpt  cptr_fastdgpt  avg_dgpt_score  avg_fdgpt_score
   0.00      v2 100.0        0.8988   0.6413      0.0292            55.0            0.0          1.0108           0.3044
   0.25      v2  98.5        0.8519   0.4930      0.0789            96.0            9.0          0.8155           0.1640
   0.50      v2  99.5        0.8317   0.4592         NaN            92.0            3.0          0.7709           0.1829
   0.75      v2  97.0        0.8548   0.5035      0.0721            96.0            8.0          0.7393           0.1852
   1.00      v2  99.0        0.8809   0.6362      0.0421            79.0            0.0          0.9095           0.2450

Saved → NLP_λ_Sweep_Checkpoints/final_results_v2.csv


In [16]:
# ── CELL 12: Diagnostic — paraphrase vs summary breakdown ────
from rouge_score import rouge_scorer as rouge_lib
 
def diagnose_v2(lam):
    lam_tag = str(lam).replace(".", "_") + "_v2"
    path = f"{SAVE_DIR}/lambda_{lam_tag}/evaded.csv"
    if not os.path.exists(path):
        print(f"λ={lam} v2: no evaded.csv")
        return
    df = pd.read_csv(path)
    df["text"] = df["text"].astype(str)
    df["original_text"] = df["original_text"].astype(str)
    sc = rouge_lib.RougeScorer(["rougeL"], use_stemmer=True)
 
    para = summ = gib = empty = 0
    for _, row in df.iterrows():
        orig, evad = row["original_text"], row["text"]
        if len(evad.strip()) < 20:
            empty += 1; continue
        alpha_ratio = sum(c.isalpha() or c.isspace() for c in evad) / max(len(evad), 1)
        junk = ("entail", "аа", "nn", "True", "False", "negative", "sentence1:")
        if evad.strip().startswith(junk) or alpha_ratio < 0.80:
            gib += 1; continue
        lr = len(evad) / max(len(orig), 1)
        if lr < 0.5:
            summ += 1
        else:
            para += 1
    total = para + summ + gib + empty
    print(f"λ={lam}: para={para}({100*para/total:.0f}%) "
          f"summ={summ}({100*summ/total:.0f}%) "
          f"gib={gib}({100*gib/total:.0f}%) "
          f"empty={empty}({100*empty/total:.0f}%)")
 
print("\n" + "="*55)
print("DIAGNOSTIC: Paraphrase vs Summary breakdown")
print("="*55)
for lam in LAMBDA_VALUES:
    diagnose_v2(lam)


DIAGNOSTIC: Paraphrase vs Summary breakdown
λ=1.0: para=131(66%) summ=66(33%) gib=3(2%) empty=0(0%)
λ=0.75: para=135(68%) summ=61(30%) gib=4(2%) empty=0(0%)
λ=0.5: para=135(68%) summ=60(30%) gib=5(2%) empty=0(0%)
λ=0.25: para=139(70%) summ=57(28%) gib=4(2%) empty=0(0%)
λ=0.0: para=124(62%) summ=74(37%) gib=2(1%) empty=0(0%)


In [17]:
# ── Print best v2 examples (strict paraphrase filter) ─────────
# Paste into a cell after the overnight run finishes.
# Shows only samples that are ACTUAL paraphrases (not summaries).

import pandas as pd
from rouge_score import rouge_scorer as rouge_lib

SAVE_DIR = "NLP_λ_Sweep_Checkpoints"

# Use the strict filter from PATCH-B
sc = rouge_lib.RougeScorer(["rougeL"], use_stemmer=True)

def is_valid_paraphrase(orig, evad):
    if len(evad.strip()) < 80 or len(orig.strip()) < 80:
        return False
    junk = ("entail", "\u0430\u0430", "nn", "True", "False",
            "negative", "sentence1:", "hypothesis:")
    if evad.strip().startswith(junk):
        return False
    if "\u0430\u0430\u0430\u0430\u0430" in evad or "nnnnn" in evad:
        return False
    alpha_ratio = sum(c.isalpha() or c.isspace() for c in evad) / len(evad)
    if alpha_ratio < 0.80:
        return False
    lr = len(evad) / max(len(orig), 1)
    if lr < 0.5 or lr > 2.0:
        return False
    rl = sc.score(orig, evad)["rougeL"].fmeasure
    if rl < 0.20 or rl > 0.85:
        return False
    return True

# Show for each lambda
for lam in [1.0, 0.75, 0.5, 0.25, 0.0]:
    lam_tag = str(lam).replace(".", "_") + "_v2"
    path = f"{SAVE_DIR}/lambda_{lam_tag}/evaded.csv"

    try:
        df = pd.read_csv(path)
    except:
        print(f"\n\u03bb={lam}: file not found"); continue

    df["text"] = df["text"].astype(str)
    df["original_text"] = df["original_text"].astype(str)

    valid = df[df.apply(lambda r: is_valid_paraphrase(r["original_text"], r["text"]), axis=1)].reset_index(drop=True)

    print(f"\n{'='*70}")
    print(f"  \u03bb={lam} v2 \u2014 Valid paraphrases: {len(valid)} / {len(df)}")
    print(f"{'='*70}")

    for i in range(min(3, len(valid))):
        orig = valid.loc[i, "original_text"]
        evad = valid.loc[i, "text"]
        rl = sc.score(orig, evad)["rougeL"].fmeasure
        lr = len(evad) / max(len(orig), 1)
        print(f"\n--- Example {i+1}  (len_ratio={lr:.2f}, rouge_l={rl:.2f}) ---")
        print(f"ORIGINAL:\n{orig[:350]}...")
        print(f"\nEVADED:\n{evad[:350]}...")
        print()


  λ=1.0 v2 — Valid paraphrases: 114 / 200

--- Example 1  (len_ratio=0.94, rouge_l=0.84) ---
ORIGINAL:
There are a few reasons why children might fall asleep in the car more easily than in other places. One reason is that the motion of the car can be soothing and help to relax the body. The sound of the car's engine and the hum of the tires on the road can also be calming and make it easier to fall asleep. The darkness inside the car can also be con...

EVADED:
There are a few reasons why children might fall asleep in the car more easily than in other places : one is that the motion of the car can be soothing and help to relax the body; the sound of the engine and the hum of the tires on the road can also be calming and make it easier to fall asleep; and the darkness inside the car may provide a sense of ...


--- Example 2  (len_ratio=0.52, rouge_l=0.62) ---
ORIGINAL:
\nFilmmakers often start their careers by working on smaller projects or by assisting on larger projects. They might 